In [26]:
"""Document chunking module for RAG system.

This module provides functionality to split documents into smaller, manageable chunks
that preserve semantic coherence and context. Supports multiple chunking strategies
including sentence-based and paragraph-based chunking with overlap.
"""

import re
from typing import List, Dict, Any
from langchain_core.documents import Document


class DocumentChunker:
    """
    A utility class for splitting documents into chunks using various strategies.

    Chunking is essential for RAG systems because:
    - Embedding models have token limits (typically 512 tokens)
    - Smaller chunks improve retrieval precision
    - Overlap preserves context across chunk boundaries

    **Chunking Strategies:**
    - Sentence-based: Splits text at sentence boundaries, respecting max chunk size
    - Paragraph-based: Splits text at paragraph boundaries
    - Overlap: Adds overlap between chunks to preserve context
    """

    def __init__(self, chunk_size: int = 300, overlap_size: int = 50, strategy: str = "sentence"):
        """
        Initializes the DocumentChunker with configuration parameters.

        Args:
            chunk_size (int): Maximum number of characters per chunk (default: 300).
            overlap_size (int): Number of characters to overlap between chunks (default: 50).
            strategy (str): Chunking strategy - "sentence" or "paragraph" (default: "sentence").

        Raises:
            ValueError: If chunk_size <= 0, overlap_size < 0, or strategy is invalid.
        """
        if chunk_size <= 0:
            raise ValueError("chunk_size must be greater than 0")
        if overlap_size < 0:
            raise ValueError("overlap_size must be non-negative")
        if strategy not in ["sentence", "paragraph"]:
            raise ValueError("strategy must be 'sentence' or 'paragraph'")

        self.chunk_size = chunk_size
        self.overlap_size = overlap_size
        self.strategy = strategy

    def chunk_document(self, document: Document) -> List[Document]:
        """
        Splits a single document into chunks based on the configured strategy.

        Args:
            document (Document): A LangChain Document object to chunk.
                Must have 'page_content' and 'metadata' attributes.

        Returns:
            List[Document]: A list of Document objects, each representing a chunk.
                Each chunk preserves the original metadata and adds:
                - 'chunk_index': Index of this chunk within the document
                - 'total_chunks': Total number of chunks for this document

        Raises:
            ValueError: If document is empty or invalid.
            TypeError: If input is not a Document object.
        """
        if not isinstance(document, Document):
            raise TypeError("Input must be a Document object.")

        if not document.page_content:
            raise ValueError("Document content is empty.")

        if not document.metadata:
            raise ValueError("Document metadata is missing.")

        if not document.metadata.get("id"):
            raise ValueError("Document metadata must contain an 'id' field.")

        # Chunk the document content based on the selected strategy
        if self.strategy == "sentence":
            chunks = self._sentence_based_chunking(document.page_content)
        elif self.strategy == "paragraph":
            chunks = self._paragraph_based_chunking(document.page_content)
        else:
            raise ValueError("Invalid chunking strategy.")

        # Create Document objects for each chunk and preserve metadata
        chunked_documents = []
        for i, chunk in enumerate(chunks):
            chunked_doc = Document(
                page_content=chunk,
                metadata={
                    "id": document.metadata["id"],
                    "title": document.metadata.get("title", ""),
                    "category": document.metadata.get("category", ""),
                    "chunk_index": i,
                    "total_chunks": len(chunks)
                }
            )
            chunked_documents.append(chunked_doc)

        return chunked_documents

    def chunk_documents(self, documents: List[Document]) -> List[Document]:
        """
        Splits multiple documents into chunks.

        Args:
            documents (List[Document]): A list of LangChain Document objects to chunk.

        Returns:
            List[Document]: A list of all chunks from all input documents.

        Raises:
            ValueError: If documents list is empty.
            TypeError: If input is not a list.
        """
        if not isinstance(documents, list):
            raise TypeError("Input must be a list of Document objects.")

        if not documents:
            raise ValueError("Documents list is empty.")

        all_chunks = []
        for document in documents:
            chunks = self.chunk_document(document)
            all_chunks.extend(chunks)

        return all_chunks

    def _sentence_based_chunking(self, text: str) -> List[str]:
        """
        Splits text into chunks at sentence boundaries.

        Process:
        1. Split text into sentences (using period, exclamation, question mark)
        2. Group sentences into chunks respecting chunk_size
        3. Add overlap between chunks

        Args:
            text (str): The text to chunk.

        Returns:
            List[str]: A list of text chunks.
        """
        sentences = re.split(r'(?<=[.!?])\s+', text.strip())
        current_chunk = sentences.pop(0) if sentences else ""
        chunks = []
        separator = ". "

        for sentence in sentences:
            if len(current_chunk) + len(sentence) + len(separator) <= self.chunk_size:
                current_chunk += separator + sentence
            else:
                chunks.append(current_chunk.strip())
                current_chunk = sentence


        if current_chunk:
            chunks.append(current_chunk.strip())

        overlap_chunks = self._overlap_chunks(chunks)

        return overlap_chunks

    def _paragraph_based_chunking(self, text: str) -> List[str]:
        """
        Splits text into chunks at paragraph boundaries.

        Process:
        1. Split text into paragraphs (using double newlines)
        2. Group paragraphs into chunks respecting chunk_size
        3. Add overlap between chunks

        Args:
            text (str): The text to chunk.

        Returns:
            List[str]: A list of text chunks.
        """
        paragraphs = text.strip().split("\n\n")
        chunks = []
        current_chunk = ""

        for paragraph in paragraphs:
            if len(current_chunk) + len(paragraph) <= self.chunk_size:
                current_chunk += "\n\n" + paragraph
            else:
                chunks.append(current_chunk.strip())
                current_chunk = paragraph

        if current_chunk:
            chunks.append(current_chunk.strip())

        overlap_chunks = self._overlap_chunks(chunks)

        return overlap_chunks


    def _overlap_chunks(self, chunks: List[str]) -> List[str]:
        """
        Adds overlap between chunks to preserve context.

        Process:
        1. Iterate through the chunks and create overlapping segments.
        2. For each chunk, include a portion of the next chunk to maintain context.

        Args:
            chunks (List[str]): A list of text chunks.

        Returns:
            List[str]: A list of text chunks with overlap.
        """
        if len(chunks) <= 1:
            return chunks

        overlap_chunks = []
        for i in range(len(chunks) - 1):
            current_overlap_size = min(self.overlap_size, len(chunks[i + 1]))
            overlap_chunk = chunks[i] + " " + chunks[i + 1][:current_overlap_size]
            overlap_chunks.append(overlap_chunk)

        return overlap_chunks

In [30]:
"""Vector store module for semantic search in RAG system.

This module provides functionality to create, manage, and query vector embeddings
for semantic search. It handles embedding generation, index creation/loading,
and similarity search operations using sentence transformers.
"""

import os
import pickle
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer


class VectorStore:
    """
    A class for creating and managing vector embeddings and indexes for semantic search.

    Responsibilities:
    - Generate embeddings from article titles and content using SentenceTransformer models.
    - Build and persist vector indexes with metadata.
    - Load pre-built indexes from disk.
    - Perform efficient similarity search using cosine similarity.
    - Manage document metadata alongside embeddings.

    **Expected Fields in DataFrame:**
    The `df` parameter passed to `create_index()` should include:
    - 'id': Unique article identifier.
    - 'title': The title of the support article (used for embeddings).
    - 'content': The content of the article.
    - 'category': Article category (e.g., 'billing', 'integrations').
    - 'last_updated': Timestamp of last update.

    **Embedding Generation:**
    Uses SentenceTransformer model `all-MiniLM-L6-v2` (384 dimensions) for embeddings.
    Embeddings should be generated from article titles (or title + content combination).

    **Similarity Search:**
    Uses cosine similarity to find the most relevant documents for a query.
    Implements brute-force approach: compute similarity with all vectors, then return top-k.
    """

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initializes the VectorStore with a specified embedding model.

        Args:
            model_name (str): Name of the SentenceTransformer model to use.
                Default: "all-MiniLM-L6-v2" (384 dimensions)
        """
        self.model_name = model_name
        self.model = None

    def _ensure_model_loaded(self):
        """Lazy load the SentenceTransformer model."""
        if self.model is None:
            self.model = SentenceTransformer(self.model_name)

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generates sentence embeddings for a list of texts using SentenceTransformer.

        Args:
            texts (List[str]): A list of text strings (e.g., article titles or content).

        Returns:
            np.ndarray: Array of sentence embedding vectors (shape: [n_texts, embedding_dim]).
                For all-MiniLM-L6-v2, embedding_dim = 384.

        Raises:
            ValueError: If `texts` is not a non-empty list.
            TypeError: If `texts` is not a list.
        """
        # Validate input
        if not isinstance(texts, list):
            raise TypeError("texts must be a list")

        if len(texts) == 0:
            raise ValueError("texts list cannot be empty")

        # Load model if needed
        self._ensure_model_loaded()

        # Generate embeddings
        embeddings = self.model.encode(texts, convert_to_numpy=True)

        return embeddings

    def create_index(self, df: pd.DataFrame, index_file_name: str, index_folder_name: str) -> Dict:
        """
        Creates a vector index from the given DataFrame and saves it to disk.

        Process:
        1. Generate embeddings for article titles (or titles + content).
        2. Create index dictionary with embeddings and metadata.
        3. Save index to disk using pickle.
        4. Return the created index.

        Args:
            df (pd.DataFrame): DataFrame containing support articles with required fields:
                - 'id': Unique article ID.
                - 'title': Article title (used for embeddings).
                - 'content': Article content.
                - 'category': Article category.
                - 'last_updated': Last updated timestamp.
            index_file_name (str): Name of the index file (e.g., 'support_index.pkl').
            index_folder_name (str): Directory where the index will be saved.

        Returns:
            dict: The created index containing:
                - 'embeddings': numpy array of embeddings (shape: [n_docs, 384])
                - 'id': list of article IDs
                - 'title': list of article titles
                - 'content': list of article contents
                - 'category': list of article categories
                - 'last_updated': list of last updated timestamps

        Raises:
            ValueError: If DataFrame is empty or missing required columns.
            OSError: If unable to create directory or save file.
        """
        # Validate DataFrame
        if not isinstance(df, pd.DataFrame):
            raise ValueError("df must be a pandas DataFrame")

        if len(df) == 0:
            raise ValueError("DataFrame is empty")

        # Validate required columns
        required_columns = ['id', 'title', 'content', 'category', 'last_updated']
        missing_columns = [col for col in required_columns if col not in df.columns]
        if missing_columns:
            raise ValueError(f"Missing required columns: {missing_columns}")

        # Generate embeddings from titles (or title + content)
        # Using title for embeddings as it's more concise and representative
        texts = df['title'].tolist()
        embeddings = self.generate_embeddings(texts)

        # Create index dictionary
        index = {
            'embeddings': embeddings,
            'id': df['id'].tolist(),
            'title': df['title'].tolist(),
            'content': df['content'].tolist(),
            'category': df['category'].tolist(),
            'last_updated': df['last_updated'].tolist()
        }

        # Create directory if it doesn't exist
        index_folder = Path(index_folder_name)
        index_folder.mkdir(parents=True, exist_ok=True)

        # Save index to disk
        index_path = index_folder / index_file_name
        with open(index_path, 'wb') as f:
            pickle.dump(index, f)

        return index

    def load_index(self, index_file_name: str, index_folder_name: str) -> Dict:
        """
        Loads a precomputed vector index from disk.

        Args:
            index_file_name (str): The file name of the saved index (e.g., 'support_index.pkl').
            index_folder_name (str): The directory where the index is stored.

        Returns:
            dict: The loaded index containing embeddings and metadata (same structure as create_index).

        Raises:
            FileNotFoundError: If the index file does not exist.
            ValueError: If the index structure is invalid or corrupted.
            KeyError: If required keys are missing in the index.
        """
        # Construct full path
        index_path = Path(index_folder_name) / index_file_name

        # Check if file exists
        if not index_path.exists():
            raise FileNotFoundError(f"Index file not found: {index_path}")

        # Load index
        try:
            with open(index_path, 'rb') as f:
                index = pickle.load(f)
        except Exception as e:
            raise ValueError(f"Failed to load index: {str(e)}")

        # Validate index structure
        required_keys = ['embeddings', 'id', 'title', 'content', 'category', 'last_updated']
        missing_keys = [key for key in required_keys if key not in index]
        if missing_keys:
            raise KeyError(f"Index missing required keys: {missing_keys}")

        # Validate embeddings shape
        if not isinstance(index['embeddings'], np.ndarray):
            raise ValueError("Index embeddings must be a numpy array")

        if len(index['embeddings']) == 0:
            raise ValueError("Index embeddings array is empty")

        # Validate metadata lengths match
        n_docs = len(index['embeddings'])
        for key in ['id', 'title', 'content', 'category', 'last_updated']:
            if len(index[key]) != n_docs:
                raise ValueError(f"Index metadata length mismatch: {key} has {len(index[key])} items, expected {n_docs}")

        return index

    def  get_query_embedding(self, query: str) -> np.ndarray:
        """
        Generates an embedding for a single query string.

        Args:
            query (str): The query string to embed.

        Returns:
            np.ndarray: The embedding vector for the query (shape: [embedding_dim]).
                For all-MiniLM-L6-v2, shape = (384,).

        Raises:
            ValueError: If `query` is not a valid non-empty string.
        """
        # Validate input
        if not isinstance(query, str):
            raise ValueError("query must be a string")

        if not query.strip():
            raise ValueError("query cannot be empty")

        # Load model if needed
        self._ensure_model_loaded()

        # Generate embedding (single text, returns 1D array)
        embedding = self.model.encode(query, convert_to_numpy=True)

        # Ensure it's 1D
        if embedding.ndim > 1:
            embedding = embedding[0]

        return embedding

    def find_top_k_matches(self, query_embedding: np.ndarray, index: Dict, k: int = 5) -> List[Tuple[int, float]]:
        """
        Finds the top-k most similar articles from the index using cosine similarity.

        Implementation should use brute-force approach:
        1. Compute cosine similarity between query_embedding and all embeddings in index
        2. Sort results by similarity score (descending)
        3. Return top-k matches

        Cosine similarity formula:
        similarity = dot(a, b) / (norm(a) * norm(b))

        Args:
            query_embedding (np.ndarray): The embedding of the input query (shape: [384]).
            index (dict): The index containing document embeddings and metadata.
                Expected structure: {'embeddings': np.ndarray, 'id': list, ...}
            k (int): The number of top matches to return (default: 5).

        Returns:
            list: A list of tuples (document_index, similarity_score) representing the top-k matches,
                  sorted by similarity score in descending order.
                  - document_index: Index position in the original DataFrame/index
                  - similarity_score: Cosine similarity score (float, typically 0.0 to 1.0)

        Raises:
            ValueError: If inputs are invalid (e.g., index missing embeddings).
            TypeError: If query_embedding is not a numpy array.
        """
        # Validate inputs
        if not isinstance(query_embedding, np.ndarray):
            raise TypeError("query_embedding must be a numpy array")

        if 'embeddings' not in index:
            raise ValueError("Index must contain 'embeddings' key")

        if not isinstance(index['embeddings'], np.ndarray):
            raise ValueError("Index embeddings must be a numpy array")

        # Get document embeddings
        doc_embeddings = index['embeddings']

        # Validate shapes
        if query_embedding.ndim != 1:
            raise ValueError("query_embedding must be 1D array")

        if doc_embeddings.ndim != 2:
            raise ValueError("doc_embeddings must be 2D array")

        if query_embedding.shape[0] != doc_embeddings.shape[1]:
            raise ValueError(f"Dimension mismatch: query has {query_embedding.shape[0]} dims, docs have {doc_embeddings.shape[1]} dims")

        # Compute cosine similarity for all documents (brute-force)
        # Cosine similarity = dot(a, b) / (norm(a) * norm(b))
        query_norm = np.linalg.norm(query_embedding)
        doc_norms = np.linalg.norm(doc_embeddings, axis=1)

        # Compute dot products
        dot_products = np.dot(doc_embeddings, query_embedding)

        # Compute cosine similarities
        similarities = dot_products / (query_norm * doc_norms)

        # Handle any NaN values (shouldn't happen, but safety check)
        similarities = np.nan_to_num(similarities, nan=0.0)

        # Get top-k indices
        # Use argpartition for efficiency with large k, but for simplicity use argsort
        top_k_indices = np.argsort(similarities)[::-1][:k]

        # Create list of (index, similarity) tuples
        results = [(int(idx), float(similarities[idx])) for idx in top_k_indices]

        return results

/Users/efloresp06/Library/Python/3.10/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [37]:
"""Retrieval system module for RAG.

This module provides functionality to retrieve relevant documents using
dense retrieval (embeddings) method.
"""

from typing import List, Tuple, Dict

# from vector_store import VectorStore


class RetrievalSystem:
    """
    A class for retrieving relevant documents using dense retrieval.

    **Retrieval Method:**
    - Dense Retrieval: Uses embeddings for semantic understanding

    **When to Use:**
    - Natural language queries, semantic understanding, synonyms
    - Finding relevant content even when exact words don't match
    """

    def __init__(self, vector_store: VectorStore):
        """
        Initializes the RetrievalSystem with a VectorStore for dense retrieval.

        Args:
            vector_store (VectorStore): An initialized VectorStore instance for dense retrieval.
        """
        self.vector_store = vector_store

    def dense_retrieve(
        self,
        query: str,
        index: Dict,
        k: int = 5
    ) -> List[Tuple[int, float]]:
        """
        Retrieves documents using dense retrieval (embeddings).

        Uses the VectorStore to:
        1. Generate query embedding
        2. Find top-k matches using cosine similarity
        3. Return ranked results

        Args:
            query (str): The search query.
            index (dict): Vector index from VectorStore containing embeddings and metadata.
            k (int): Number of top results to return (default: 5).

        Returns:
            List[Tuple[int, float]]: List of (chunk_index, similarity_score) tuples,
                sorted by similarity score (descending).

        Raises:
            ValueError: If query is empty or index is invalid.
        """
        if not isinstance(query, str):
            raise TypeError("Query must be a string.")

        query = query.strip()
        if not query:
            raise ValueError("Query cannot be empty.")

        if not isinstance(index, dict):
            raise TypeError("Index must be a dictionary.")

        if "embeddings" not in index:
            raise ValueError("Index is missing the 'embeddings' key.")

        query_embedding = self.vector_store.get_query_embedding(query)

        return self.vector_store.find_top_k_matches(
            query_embedding,
            index,
            k,
        )

    def retrieve(
        self,
        query: str,
        index: Dict,
        k: int = 5
    ) -> List[Tuple[int, float]]:
        """
        Retrieves documents using dense retrieval.

        This is a unified interface that uses dense retrieval.

        Args:
            query (str): The search query.
            index (dict): Vector index for dense retrieval.
            k (int): Number of top results to return (default: 5).

        Returns:
            List[Tuple[int, float]]: List of (chunk_index, similarity_score) tuples,
                sorted by similarity score (descending).

        Raises:
            ValueError: If query is empty or index is invalid.
        """
        return self.dense_retrieve(query, index, k)

rs = RetrievalSystem(vector_store=VectorStore())
sample_embeddings = np.random.rand(10, 384)  # 10 documents with 384-dimensional embeddings
sample_dataframe = pd.DataFrame({
    'id': range(10),
    'title': [f"Document {i}" for i in range(10)],
    'content': [f"This is the content of document {i}." for i in range(10)],
    'category': [f"Category {i % 3}" for i in range(10)],
    'last_updated': pd.date_range(start='2023-01-01', periods=10, freq='D')
})

sample_index = {
    'embeddings': sample_embeddings,
    'id': sample_dataframe['id'].tolist(),
    'title': sample_dataframe['title'].tolist(),
    'content': sample_dataframe['content'].tolist(),
    'category': sample_dataframe['category'].tolist(),
    'last_updated': sample_dataframe['last_updated'].tolist()
}

rs.retrieve("Hola", sample_index, k=3)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6724.99it/s]


[(3, 0.07364092403273825), (7, 0.051957503012806765), (6, 0.05003257411972154)]

In [ ]:
"""Document loading module for RAG system.

This module provides functionality to load support article data from JSON files
and convert it into formats suitable for RAG processing, including pandas
DataFrames and LangChain Document objects.
"""

import json
from pathlib import Path
from typing import List

import pandas as pd
from langchain_core.documents import Document


class DocumentLoader:
    """
    A utility class for loading and converting JSON support article data into LangChain Document objects.

    Input Requirement:
    - The input must be a JSON file with a list of support articles.
    - Each article must contain the following fields:
        - 'id': Unique identifier for each article.
        - 'title': The title of the support article.
        - 'content': The main content or body of the article.
        - 'category': Category of the article (e.g., 'billing', 'integrations', 'troubleshooting').
        - 'last_updated': Timestamp indicating when the article was last updated.

    Preferred date format for 'last_updated': 'YYYY-MM-DD' or 'YYYY-MM-DD HH:MM:SS'.

    Example JSON format:
    [
        {
            "id": "KB001",
            "title": "How to request a refund",
            "content": "To request a refund, navigate to your account settings...",
            "category": "billing",
            "last_updated": "2024-10-05"
        }
    ]
    """

    def __init__(self, json_file: str):
        """
        Initializes the DocumentLoader with the path to a JSON file.

        Args:
            json_file (str): Absolute or relative path to the JSON file.

        Raises:
            ValueError: If input is not a non-empty string.
        """
        if not isinstance(json_file, str) or not json_file.strip():
            raise ValueError("json_file must be a non-empty string")
        self.json_file = json_file

    def load_data(self) -> pd.DataFrame:
        """
        Loads and parses support article data from the specified JSON file.

        Functionality:
        - Verifies that the file exists and is accessible.
        - Parses the JSON content into a pandas DataFrame.
        - Validates that each article contains all required fields.
        - Returns a DataFrame with all article data.

        Returns:
            pd.DataFrame: A DataFrame containing all support articles with required columns:
                - 'id': Article identifier
                - 'title': Article title
                - 'content': Article content
                - 'category': Article category
                - 'last_updated': Last update date

        Raises:
            FileNotFoundError: If the file path does not exist.
            json.JSONDecodeError: If file contains invalid JSON.
            ValueError: If required fields are missing or DataFrame is empty.
            KeyError: If required columns are not present in the data.
        """
        # Check if file exists
        file_path = Path(self.json_file)
        if not file_path.exists():
            raise FileNotFoundError(f"File not found: {self.json_file}")

        # Load JSON
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
        except json.JSONDecodeError as e:
            raise json.JSONDecodeError(f"Invalid JSON in file: {e.msg}", e.doc, e.pos)

        # Validate data is a list
        if not isinstance(data, list):
            raise ValueError("JSON file must contain a list of articles")

        # Validate data is not empty
        if len(data) == 0:
            raise ValueError("JSON file contains no articles")

        # Convert to DataFrame
        df = pd.DataFrame(data)

        # Validate required columns
        required_columns = ['id', 'title', 'content', 'category', 'last_updated']
        missing_columns = [col for col in required_columns if col not in df.columns]
        if missing_columns:
            raise KeyError(f"Missing required columns: {missing_columns}")

        # Validate no missing values in required fields
        for col in required_columns:
            if df[col].isna().any():
                raise ValueError(f"Column '{col}' contains missing values")

        # Validate DataFrame is not empty after cleaning
        if len(df) == 0:
            raise ValueError("DataFrame is empty after validation")

        return df

    def create_documents(self, data: pd.DataFrame) -> List[Document]:
        """
        Converts validated article data from a DataFrame into LangChain Document objects.

        Args:
            data (pandas.DataFrame): A DataFrame where each row represents a support article. Required columns:
                - 'id'
                - 'title'
                - 'content'
                - 'category'
                - 'last_updated'

        Returns:
            List[Document]: A list of LangChain-compatible Document objects with metadata.
                Each Document should have:
                - page_content: The article content (or title + content)
                - metadata: Dictionary containing 'id', 'title', 'category', 'last_updated'

        Raises:
            ValueError: If required columns are missing or DataFrame is empty.
            TypeError: If input is not a pandas DataFrame.
        """
        # Validate input type
        if not isinstance(data, pd.DataFrame):
            raise TypeError("Input must be a pandas DataFrame")

        # Validate DataFrame is not empty
        if len(data) == 0:
            raise ValueError("DataFrame is empty")

        # Validate required columns
        required_columns = ['id', 'title', 'content', 'category', 'last_updated']
        missing_columns = [col for col in required_columns if col not in data.columns]
        if missing_columns:
            raise ValueError(f"Missing required columns: {missing_columns}")

        # Create Document objects
        documents = []
        for _, row in data.iterrows():
            # Use content as page_content (or title + content)
            page_content = f"{row['title']}\n\n{row['content']}"

            # Create metadata dictionary
            metadata = {
                'id': row['id'],
                'title': row['title'],
                'category': row['category'],
                'last_updated': row['last_updated']
            }

            # Create LangChain Document
            doc = Document(page_content=page_content, metadata=metadata)
            documents.append(doc)

        return documents

In [ ]:
"""RAG pipeline module for complete retrieval-augmented generation system.

This module integrates all components (document loading, chunking, retrieval, and response generation)
into a complete RAG pipeline.
"""

import pandas as pd
from typing import List, Dict, Any, Optional
from langchain_core.documents import Document

from document_loader import DocumentLoader
from vector_store import VectorStore
from document_chunker import DocumentChunker
from retrieval_system import RetrievalSystem


class RAGPipeline:
    """
    A complete RAG pipeline that orchestrates document processing, chunking, retrieval, and response generation.

    **Pipeline Flow:**
    1. Load documents from JSON file
    2. Chunk documents into smaller pieces
    3. Build dense index (embeddings)
    4. Retrieve relevant chunks for queries using dense retrieval
    5. Generate responses from retrieved context
    """

    def __init__(
        self,
        document_loader: DocumentLoader,
        vector_store: VectorStore,
        document_chunker: DocumentChunker,
        retrieval_system: RetrievalSystem
    ):
        """
        Initializes the RAG pipeline with all required components.

        Args:
            document_loader (DocumentLoader): Loader for reading documents from JSON.
            vector_store (VectorStore): Store for generating embeddings and dense retrieval.
            document_chunker (DocumentChunker): Chunker for splitting documents.
            retrieval_system (RetrievalSystem): System for retrieving relevant chunks.
        """
        self.document_loader = document_loader
        self.vector_store = vector_store
        self.document_chunker = document_chunker
        self.retrieval_system = retrieval_system

        self.documents = []
        self.chunks = []
        self.vector_index = None

    def build_index(
        self,
        index_file_name: str = "support_index.pkl",
        index_folder_name: str = "index",
    ) -> Dict[str, Any]:
        """
        Builds the complete vector index for the RAG system.

        Returns:
            Dict[str, Any]: The created vector index.

        Raises:
            ValueError: If documents cannot be loaded or processed.
            OSError: If the index cannot be saved.
        """
        df = self.document_loader.load_data()

        try:
            self.documents = self.document_loader.create_documents(df)
            self.chunks = self.document_chunker.chunk_documents(self.documents)
        except Exception as e:
            raise ValueError(f"Error during document loading or chunking: {e}")

        chunk_df = pd.DataFrame({
            "id": [doc.metadata.get("id") for doc in self.chunks],
            "title": [doc.metadata.get("title") for doc in self.chunks],
            "content": [doc.page_content for doc in self.chunks],
            "category": [doc.metadata.get("category") for doc in self.chunks],
            "last_updated": [doc.metadata.get("last_updated") for doc in self.chunks]
        })

        self.vector_index = self.vector_store.create_index(
            chunk_df,
            index_file_name,
            index_folder_name,
        )

        return self.vector_index

    def load_index(self, index_file_name: str = "support_index.pkl", index_folder_name: str = "index"):
        """
        Loads a pre-built index from disk.

        Args:
            index_file_name (str): Name of the index file to load.
            index_folder_name (str): Directory where the index is stored.

        Returns:
            dict: The loaded vector index.

        Raises:
            FileNotFoundError: If index file does not exist.
        """
        try:
            self.vector_index = self.vector_store.load_index(
                index_file_name,
                index_folder_name,
            )
        except FileNotFoundError:
            raise
        except ValueError:
            raise
        except Exception as exc:
            raise ValueError(
                f"Failed to load vector index from "
                f"'{index_folder_name}/{index_file_name}'."
            ) from exc

        return self.vector_index

    def query(
        self,
        question: str,
        top_k: int = 3
    ) -> Dict[str, Any]:
        """
        Processes a query and returns a response with relevant context.

        Process:
        1. Retrieve relevant chunks using dense retrieval
        2. Combine chunks into context
        3. Generate response from context

        Args:
            question (str): The user's question or query.
            top_k (int): Number of top chunks to retrieve (default: 3).

        Returns:
            dict: Response dictionary containing:
                - 'answer': Generated answer text
                - 'sources': List of source document IDs
                - 'chunks': List of retrieved chunk texts
                - 'similarity_scores': List of similarity scores

        Raises:
            ValueError: If question is empty or index not built.
        """
        if not question.strip():
            raise ValueError("Question cannot be empty.")

        if self.vector_index is None:
            raise ValueError("Vector index is not built or loaded.")

        # Retrieve top-k relevant chunks
        retrieved = self.retrieval_system.retrieve(question, self.vector_index, k=top_k)
        chunk_indices, similarity_scores = zip(*retrieved) if retrieved else ([], [])

        # Create context from retrieved chunks
        context = self._create_context(list(chunk_indices), list(similarity_scores))

        # Generate response from context
        answer = self._generate_response(question, context)

        # Prepare sources and chunks for output
        sources = [self.vector_index['id'][idx] for idx in chunk_indices]
        chunks_texts = [self.vector_index['content'][idx] for idx in chunk_indices]

        return {
            "answer": answer,
            "sources": sources,
            "chunks": chunks_texts,
            "similarity_scores": list(similarity_scores)
        }

    def _create_context(self, chunk_indices: List[int], similarity_scores: List[float]) -> str:
        """
        Combines retrieved chunks into a coherent context string.

        Args:
            chunk_indices (List[int]): Indices of chunks to include.
            similarity_scores (List[float]): Similarity scores for each chunk.

        Returns:
            str: Combined context string from retrieved chunks.
        """
        context_parts = []
        for idx, score in zip(chunk_indices, similarity_scores):
            chunk_content = self.vector_index['content'][idx]
            context_parts.append(f"[Score: {score:.4f}] {chunk_content}")

        return "\n\n".join(context_parts)

    def _generate_response(self, question: str, context: str) -> str:
        """
        Generates a response from the question and retrieved context.

        For this challenge, create a simple response that:
        - Acknowledges the question
        - Summarizes the relevant information from context
        - No LLM required (simple text processing)

        Args:
            question (str): The user's question.
            context (str): Retrieved context from chunks.

        Returns:
            str: Generated response text.
        """
        # Simple response generation: acknowledge question and summarize context
        response = f"Question: {question}\n\n"

        if context:
            response += "Based on the retrieved information, here are the relevant details:\n\n"
            response += context
        else:
            response += "We could not find any relevant information."

        return response

